# 04 — Valeur de chaque emplacement candidat (Estimer)

**Rôle dans le pipeline :** donner à chaque emplacement possible une **valeur** = nombre d'habitants qu'une station y servirait, **pondérés par leur manque de bus**.

| Lit | Produit |
|---|---|
| `silver/carreaux_acces.parquet` | `silver/emplacements.parquet` |

**Emplacements candidats :** le centre de chaque carreau habité.

**En 3 temps :**
1. **Besoin** d'un carreau = habitants × poids selon son niveau d'offre de bus (`POIDS_BESOIN`).
2. **Zone servie** par une station = carreaux à moins de `RAYON_STATION_M` (500 m).
3. **Valeur** de l'emplacement = somme des besoins de sa zone.

**À dire honnêtement au jury :** c'est une **estimation par règles**, explicable ligne à ligne, **pas un modèle prédictif** de l'usage réel.

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from parametres import SILVER, COMMUNES, POIDS_BESOIN, RAYON_STATION_M, RAYON_TERRE_M, NIVEAUX_OFFRE

carreaux = pd.read_parquet(SILVER / "carreaux_acces.parquet")

### 1. Besoin de chaque carreau

| Niveau d'offre | Poids | Pourquoi |
|---|---|---|
| Aucun arrêt | 1,0 | Besoin maximal |
| Très faible | 1,0 | Bus quasi inexistant |
| Faible | 0,6 | Bus peu fréquent |
| Correcte | 0,3 | Besoin réduit, mais réel pour les trajets occasionnels |
| Forte | 0,1 | Le bus suffit souvent ; petit intérêt pour l'intermodalité |

Ce sont des **choix**, écrits en clair dans `parametres.py` et modifiables pour tester d'autres stratégies. Le contrôle `assert` garantit qu'aucun niveau n'a été oublié dans le dictionnaire.

In [2]:
carreaux["poids_besoin"] = carreaux["niveau_offre"].map(POIDS_BESOIN)
assert carreaux["poids_besoin"].notna().all(), "un niveau d'offre n'a pas de poids"
carreaux["pop_ponderee"] = carreaux["ind"] * carreaux["poids_besoin"]

besoin = carreaux.groupby("niveau_offre").agg(habitants=("ind", "sum"), besoin=("pop_ponderee", "sum")).reindex(NIVEAUX_OFFRE)
besoin["part_du_besoin_%"] = (100 * besoin["besoin"] / besoin["besoin"].sum()).round(1)
besoin.round()

,habitants,besoin,part_du_besoin_%
niveau_offre,,,
Aucun arrêt,9728.0,9728.0,15.0
Très faible,7452.0,7452.0,11.0
Faible,34173.0,20504.0,31.0
Correcte,59109.0,17733.0,27.0
Forte,108944.0,10894.0,16.0


### 2 et 3. Zone servie et valeur de chaque emplacement

Cette fois, l'arbre range les **carreaux** (et non les arrêts) : pour chaque candidat, on retrouve tous les carreaux à moins de 500 m.

- `pop_couverte` : habitants **bruts** servis ;
- `valeur` : habitants **pondérés par le besoin**.

In [3]:
points = np.radians(carreaux[["lat", "lon"]].values)
couverts = BallTree(points, metric="haversine").query_radius(points, r=RAYON_STATION_M / RAYON_TERRE_M)

ind = carreaux["ind"].values
pond = carreaux["pop_ponderee"].values
carreaux["pop_couverte"] = [ind[v].sum() for v in couverts]
carreaux["valeur"] = [pond[v].sum() for v in couverts]

### Top 10, sans tenir compte des chevauchements

Le top 10 sera **groupé** : des carreaux voisins partagent presque les mêmes habitants. Y mettre 10 stations serait absurde (elles couvriraient les mêmes personnes). C'est ce que règle le notebook **05** : une fois une station choisie, ses habitants ne comptent plus pour les suivantes.

In [4]:
top = carreaux.sort_values("valeur", ascending=False).head(10)
top.assign(commune=top["commune"].map(COMMUNES))[
    ["idcar_200m", "commune", "lat", "lon", "pop_couverte", "valeur", "niveau_offre"]
].round(3)

,idcar_200m,commune,lat,lon,pop_couverte,valeur,niveau_offre
1629,CRS2975RES200mN7670000E315800,Saint-Paul,-21.061,55.228,2501.5,1545.55,Faible
3296,CRS2975RES200mN7681800E326000,La Possession,-20.955,55.327,4074.0,1544.40,Faible
1157,CRS2975RES200mN7666200E317200,Saint-Paul,-21.095,55.241,2186.5,1526.90,Aucun arrêt
1630,CRS2975RES200mN7670000E316000,Saint-Paul,-21.061,55.230,2421.5,1514.35,Faible
1158,CRS2975RES200mN7666200E317400,Saint-Paul,-21.095,55.243,1903.0,1506.75,Aucun arrêt
3276,CRS2975RES200mN7681600E326200,La Possession,-20.957,55.329,4545.5,1503.25,Correcte
1186,CRS2975RES200mN7666400E317200,Saint-Paul,-21.093,55.241,2061.5,1451.95,Aucun arrêt
1185,CRS2975RES200mN7666400E317000,Saint-Paul,-21.093,55.239,2099.5,1427.50,Correcte
3295,CRS2975RES200mN7681800E325800,La Possession,-20.955,55.325,3584.5,1402.80,Faible
1606,CRS2975RES200mN7669800E315800,Saint-Paul,-21.063,55.228,2008.0,1402.20,Faible


In [5]:
carreaux.to_parquet(SILVER / "emplacements.parquet", index=False)
print("Écrit :", SILVER / "emplacements.parquet")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\data\silver\emplacements.parquet
